In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gtda.time_series import SingleTakensEmbedding, takens_embedding_optimal_parameters
from sklearn.decomposition import PCA
from gtda.plotting import plot_point_cloud
from gtda.diagrams import PersistenceEntropy, PairwiseDistance
from gtda.diagrams import Silhouette
from gtda.diagrams import BettiCurve 
from gtda.homology import VietorisRipsPersistence
from gtda.time_series import TakensEmbedding, SlidingWindow
import plotly.express as px
from sklearn.metrics import mutual_info_score
from gtda.metaestimators import CollectionTransformer
from gtda.pipeline import Pipeline
from gtda.plotting import plot_heatmap

In [ ]:
slug_17df = pd.read_csv('processed_AA-17.csv')
slug_17ts = slug_17df[['TimeStamp', 'TH_Pressure']]
slug_17ts['TimeStamp'] = pd.to_datetime(slug_17ts['TimeStamp'])

slug_july = slug_17ts[(slug_17ts['TimeStamp'] > '2022-07-19 00:00:00') & (slug_17ts['TimeStamp'] < '2022-07-20 12:00:00')]

from scipy.interpolate import CubicSpline

xmin = 1
xmax = len(slug_july['TH_Pressure'])-1
some_step = 0.1                         # this is a 6 seconds step, 10 points per minute
cs = CubicSpline(range(len(slug_july)), slug_july['TH_Pressure'])
x_range = np.arange(xmin, xmax, some_step)

slug_july.insert(1, "idx", np.linspace(0, len(slug_july)-1, num=len(slug_july)), True)

print(cs(x_range).shape)

# upsampled to 10 times

new_range = pd.date_range(slug_july['TimeStamp'].iloc[0], slug_july['TimeStamp'].iloc[-1], freq='6S')
new_range = new_range[0:len(cs(x_range))]

slug_july_spline = []
slug_july_spline = pd.DataFrame(data=cs(x_range)) 
slug_july_spline['TimeStamp'] = new_range


figure_FHP = px.scatter()
figure_FHP.add_scatter(x=slug_july_spline["TimeStamp"], y=slug_july_spline[0],  name="FHP_wellAA17")
figure_FHP.show()

In [ ]:
from gtda.time_series import SlidingWindow, TakensEmbedding

signal = slug_july_spline[0]
print(len(signal))

window_size = 300 
window_stride = 200
SW = SlidingWindow(size=window_size, stride=window_stride)

X_windows = SW.fit_transform(signal)

In [ ]:
optimal_time_delay = 20
optimal_embedding_dimension = 10
stride = 1

from gtda.pipeline import Pipeline
from gtda.metaestimators import CollectionTransformer
from gtda.diagrams import PersistenceEntropy

TE = TakensEmbedding(time_delay=optimal_time_delay, dimension=optimal_embedding_dimension, stride=1)
homology_dimensions = (0, 1)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)

slugging_signals = X_windows
slugging_point_cloud  = TE.fit_transform(slugging_signals)
slugging_diagrams = VRP.fit_transform(slugging_point_cloud)

In [ ]:
p_W = 2
PD = PairwiseDistance(metric='wasserstein',
                      metric_params={'p': p_W, 'delta': 0.1},
                      order=None)

X300_distance_W = PD.fit_transform(slugging_diagrams)
plot_heatmap(X300_distance_W[:, :, 0], colorscale='blues')

In [ ]:
plot_heatmap(X300_distance_W[:, :, 1], colorscale='blues')

In [ ]:
figure_W2 = px.scatter()
figure_W2.add_scatter(y=X300_distance_W[1,:,0],  name="W2 homology dim 0")
figure_W2.add_scatter(y=X300_distance_W[1,:,1],  name="W2 homology dim 1")
figure_W2.show()

In [ ]:
x = X_windows[0]
bins = np.linspace(6.2, 7.2, 50)

fig, ax = plt.subplots(1, 2, figsize=(12, 4),
                       sharex=True, sharey=True,
                       subplot_kw={'xlim':(6.2, 7.2)#,
                                   #'ylim':(-0.02, 120)
                                   })
fig.subplots_adjust(wspace=0.05)
for i, offset in enumerate([0.0, 0.6]):
    ax[i].hist(x, bins=bins + offset)#, normed=True)
    ax[i].plot(x, np.full_like(x, 0.1), '|k',
               markeredgewidth=1)

In [ ]:
from scipy.stats import norm

x_d = np.linspace(3.5, 13.5, 500)
density = sum(norm(xi).pdf(x_d) for xi in x)

plt.fill_between(x_d, density, alpha=0.5)
plt.plot(x, np.full_like(x, -0.1), '|k', markeredgewidth=1)

#plt.axis([6.2, 7.7, -0.2, 500])

In [ ]:
import seaborn as sns

sns.distplot(x, bins=50, kde=True)

In [ ]:
from sklearn.neighbors import KernelDensity

x = X_windows[0]

# instantiate and fit the KDE model
kde = KernelDensity(bandwidth=9.0E-2, kernel='gaussian')
kde.fit(x[:, None])

# score_samples returns the log of the probability density
logprob = kde.score_samples(x_d[:, None])

plt.fill_between(x_d, np.exp(logprob), alpha=0.5)
plt.plot(x, np.full_like(x, -0.01), '|k', markeredgewidth=1)
#plt.ylim(-0.02, 0.22)

In [ ]:
x = X_windows[64]

# instantiate and fit the KDE model
kde = KernelDensity(bandwidth=9.0E-2, kernel='gaussian')
kde.fit(x[:, None])

# score_samples returns the log of the probability density
logprob = kde.score_samples(x_d[:, None])

plt.fill_between(x_d, np.exp(logprob), alpha=0.5)
plt.plot(x, np.full_like(x, -0.01), '|k', markeredgewidth=1)

In [ ]:
x = X_windows[33]

# instantiate and fit the KDE model
kde = KernelDensity(bandwidth=9.0E-2, kernel='gaussian')
kde.fit(x[:, None])

# score_samples returns the log of the probability density
logprob_33 = kde.score_samples(x_d[:, None])

plt.fill_between(x_d, np.exp(logprob_33), alpha=0.5)
plt.plot(x, np.full_like(x, -0.01), '|k', markeredgewidth=1)

In [ ]:
sns.distplot(x, bins=50, kde=True)

In [ ]:
figure = px.scatter()
figure.add_scatter(x=x_d, y=np.exp(logprob),  name="kde")
figure.add_scatter(x=x_d, y=np.exp(logprob_33),  name="kde")

figure.show()


In [ ]:
# calculate the kl divergence
from math import log2

def kl_divergence(p, q):
 return sum(p[i] * log2(p[i]/q[i]) for i in range(len(p)))

In [ ]:
kl_divergence(logprob, logprob_33)

In [ ]:
from scipy.stats import wasserstein_distance
wasserstein_distance(logprob, logprob_33)

In [ ]:
x_d = np.linspace(3.5, 13.5, 500)
kde = KernelDensity(bandwidth=9.0E-2, kernel='gaussian')

distance_K = (len(X_windows),len(X_windows))
distance_K = np.zeros(distance_K)

for i in range(len(X_windows)):

    # instantiate and fit the KDE model
    kde.fit(X_windows[i][:, None])
    logprob_i = kde.score_samples(x_d[:, None])

    for j in range(len(X_windows)):
    
        # instantiate and fit the KDE model
        kde.fit(X_windows[j][:, None])
        logprob_j = kde.score_samples(x_d[:, None])
        temp = wasserstein_distance(logprob_i, logprob_j)
        distance_K[i,j] = temp

In [ ]:
plot_heatmap(distance_K, colorscale='blues')

In [ ]:
figure_W2 = px.scatter()
figure_W2.add_scatter(y=X300_distance_W[1,:,0],  name="W2 homology dim 0")
figure_W2.add_scatter(y=X300_distance_W[1,:,1],  name="W2 homology dim 1")
figure_W2.add_scatter(y=distance_K[1,:]/100,  name="Kantorovic")

figure_W2.show()